In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import aostools_functions as ac

# 1. 打开并重命名坐标
era5_file = "/home/weiji/restart_exam/era5_data/Era5_uvt_2015to2024.nc"
ds = (
    xr.open_dataset(era5_file)
      .rename({
          "valid_time": "time",
          "pressure_level": "plev",
          "latitude": "lat",
          "longitude": "lon",
      })
)



# 2. 提取坐标和场变量
lat  = ds.lat.values
plev = ds.plev.values
u    = ds.u.values
v    = ds.v.values
t    = ds.t.values

# 3. 设置输出目录
outdir = "/home/weiji/restart_exam/plots/epflux_speedtest/era5_vectoronly"
os.makedirs(outdir, exist_ok=True)

# 4. 设置波数组合列表
wave_sets = [
    [1],
    [2],
    [3],
    [1, 2]
]

# 5. 遍历每组波数
for waves in wave_sets:
    tag = "_".join([f"wave{k}" for k in waves])
    print(f"Processing {tag}...")

    ep1_sum = np.zeros((u.shape[0], len(plev), len(lat)))
    ep2_sum = np.zeros_like(ep1_sum)

    for k in waves:
        ep1_k, ep2_k, *_ = ac.ComputeEPfluxDiv(
            lat, plev, u, v, t,
            wave=k,
            do_ubar=True
        )
        ep1_sum += ep1_k
        ep2_sum += ep2_k

    ep1_mean = np.mean(ep1_sum, axis=0)
    ep2_mean = np.mean(ep2_sum, axis=0)

    f1 = xr.DataArray(ep1_mean, coords={'plev': plev, 'lat': lat}, dims=['plev', 'lat']).transpose('lat', 'plev')
    f2 = xr.DataArray(ep2_mean, coords={'plev': plev, 'lat': lat}, dims=['plev', 'lat']).transpose('lat', 'plev')

    # 去掉最底层
    max_plev = float(f1.plev.max())
    f1_plot = f1.sel(plev=f1.plev.where(f1.plev < max_plev, drop=True))
    f2_plot = f2.sel(plev=f2.plev.where(f2.plev < max_plev, drop=True))

    # 下采样（可选）
    step_lat = 10
    step_plev = 2
    f1_sub = f1_plot.isel(lat=slice(None, None, step_lat), plev=slice(None, None, step_plev))
    f2_sub = f2_plot.isel(lat=slice(None, None, step_plev))

    # 绘图：100–1 hPa
    fig, ax = plt.subplots(figsize=(8, 6))
    ac.PlotEPfluxArrows(
        f1_sub.lat, f1_sub.plev,
        f1_sub, f2_sub,
        fig, ax,
        xlim=[-90, 90],
        ylim=[100, 1],
        yscale='log',
        scale=1e16
    )
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(f"EP-Flux (waves {waves})\nERA5 2015–2024, 100–1 hPa, U/V/T only, do_ubar=True (with shear correction)")

    outfile = os.path.join(
        outdir,
        f"epflux_{tag}_ERA5_2015to2024_100to1hPa_withshear.png"
    )
    fig.savefig(outfile, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Saved: {outfile}")


In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import aostools_functions as ac

# 1. 打开并重命名坐标
era5_file = "/home/weiji/restart_exam/era5_data/Era5_uvt_2015to2024.nc"
ds = (xr.open_dataset(era5_file)
       .rename({'valid_time':'time',
                'pressure_level':'plev',
                'latitude':'lat',
                'longitude':'lon'}))

# 2. 提取坐标和场变量
lat  = ds.lat.values       # [nlat]
plev = ds.plev.values      # [nlev]
u    = ds.u.values         # [time,plev,lat,lon]
v    = ds.v.values
t    = ds.t.values

# 3. 设置输出目录（带散度的版本）
outdir = "/home/weiji/restart_exam/plots/epflux_speedtest/era5_vectorwithcontour"
os.makedirs(outdir, exist_ok=True)

# 4. 波数组合列表
wave_sets = [
    [1],
    [2],
    [3],
    [1, 2]
]

# 5. 遍历每组波数，计算 EP-flux、散度，绘制矢量 + 散度
for waves in wave_sets:
    tag = "_".join(f"wave{k}" for k in waves)
    print(f"Processing {tag}...")

    nt, npv, nlt = u.shape[0], len(plev), len(lat)

    # 累加 EP‐flux 分量和散度（逐时刻计算 -> 再时间平均：物理上正确）
    ep1_sum  = np.zeros((nt, npv, nlt))
    ep2_sum  = np.zeros_like(ep1_sum)
    div_sum  = np.zeros_like(ep1_sum)

    for k in waves:
        ep1_k, ep2_k, d1_k, d2_k = ac.ComputeEPfluxDiv(
            lat, plev, u, v, t,
            wave=k,
            do_ubar=False
        )
        ep1_sum += ep1_k
        ep2_sum += ep2_k
        div_sum += (d1_k + d2_k)

    # 时间平均（得到 [plev, lat]）
    ep1_m = ep1_sum.mean(axis=0)
    ep2_m = ep2_sum.mean(axis=0)
    div_m = div_sum.mean(axis=0)

    # 包装为 DataArray 并转置到 [lat, plev]
    F1 = xr.DataArray(ep1_m,
                      coords={'plev': plev, 'lat': lat},
                      dims=['plev','lat']).transpose('lat','plev')
    F2 = xr.DataArray(ep2_m,
                      coords={'plev': plev, 'lat': lat},
                      dims=['plev','lat']).transpose('lat','plev')
    D  = xr.DataArray(div_m,
                      coords={'plev': plev, 'lat': lat},
                      dims=['plev','lat']).transpose('lat','plev')

    # 下采样：每 10°、每 2 层（先对全层下采样，供矢量使用）
    step_lat, step_plev = 10, 2
    F1s = F1.isel(lat=slice(None,None,step_lat),
                  plev=slice(None,None,step_plev))
    F2s = F2.isel(lat=slice(None,None,step_lat),
                  plev=slice(None,None,step_plev))
    Ds  = D.isel(lat=slice(None,None,step_lat),
                 plev=slice(None,None,step_plev))

    # === 去掉最底层 (最大气压层) —— 仅用于“矢量”，散度仍画全层 ===
    max_plev_sub = float(F1s.plev.max())
    F1s = F1s.sel(plev=F1s.plev.where(F1s.plev < max_plev_sub, drop=True))
    F2s = F2s.sel(plev=F2s.plev.where(F2s.plev < max_plev_sub, drop=True))
    # 如果你也想让等值线/填色同步去掉底层，可对 Ds 做同样处理；这里按你的要求保留全层用于散度。

    # 构造用于矢量的 2D 网格（与 ep1/ep2 大小匹配）
    X_sub, Y_sub = np.meshgrid(F1s.lat.values, F1s.plev.values)
    ep1_arr = F1s.values.T   # (plev_sub, lat_sub)
    ep2_arr = F2s.values.T

    # 散度用全分辨率（非下采样、含底层）来画
    LAT2D, P2D = np.meshgrid(lat, plev)

    # 绘图
    fig, ax = plt.subplots(figsize=(8,6))

    # —— 散度填色（-5..5，共41级；单位 m/s/day）
    cf = ax.contourf(
        LAT2D, P2D, D.T,
        levels=np.linspace(-5, 5, 41),
        cmap='RdBu_r',
        extend='both'
    )
    cbar = fig.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label('EP flux divergence (m/s/day)')

    # —— 散度等值线（-5..5, 间隔 1）
    cs = ax.contour(
        LAT2D, P2D, D.T,
        levels=np.arange(-5, 6, 1),
        colors='k',
        linewidths=0.5
    )
    ax.clabel(cs, fmt='%d', inline=True, fontsize=8)

    # —— 矢量箭头（完全复用原 PlotEPfluxArrows；此处传入已去底层的下采样结果）
    ac.PlotEPfluxArrows(
        X_sub, Y_sub,
        ep1_arr, ep2_arr,
        fig, ax,
        xlim=[-90, 90],
        ylim=[400, 10],
        xscale='linear',
        yscale='log',
        invert_y=True,    # 让 ylim 决定方向
        pivot='middle',
        scale=1e16
    )

    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(
        f"EP-flux & divergence (waves={waves})\n"
        "ERA5 2015–2024 UVT, full period, do_ubar=False"
    )

    # 保存
    fname = f"epflux_{tag}_ERA5_2015to2024_full_noshear_div.png"
    outfile = os.path.join(outdir, fname)
    fig.tight_layout()
    fig.savefig(outfile, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Saved: {outfile}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import aostools_functions as ac

# ----------------------------
# 通用绘图函数（全分辨率散度 + 24×12 箭头）
# ----------------------------
def plot_vector_with_contour(lat, plev_hpa, ep1, ep2, d1, d2, outdir, tag):
    print(f"\n--- Plotting {tag} ---")
    print(" lat.shape:", lat.shape, " plev_hpa.shape:", plev_hpa.shape)

    # 时间平均
    ep1_m = ep1.mean(axis=0)
    ep2_m = ep2.mean(axis=0)
    div_m = (d1 + d2).mean(axis=0)
    print(" ep1_m min/max:", np.nanmin(ep1_m), np.nanmax(ep1_m))
    print(" ep2_m min/max:", np.nanmin(ep2_m), np.nanmax(ep2_m))
    print(" div_m min/max:", np.nanmin(div_m), np.nanmax(div_m))

    # 包装并转 (lat,plev)
    F1 = xr.DataArray(ep1_m,
                      coords={'plev':plev_hpa,'lat':lat},
                      dims=['plev','lat']).transpose('lat','plev')
    F2 = xr.DataArray(ep2_m,
                      coords={'plev':plev_hpa,'lat':lat},
                      dims=['plev','lat']).transpose('lat','plev')
    D  = xr.DataArray(div_m,
                      coords={'plev':plev_hpa,'lat':lat},
                      dims=['plev','lat']).transpose('lat','plev')

    # 绘图：全场散度
    fig, ax = plt.subplots(figsize=(10,6))
    ax.set_xlim(-90, 90)
    ax.set_ylim(400, 10)
    ax.set_yscale('log')
    LAT2D, P2D = np.meshgrid(lat, plev_hpa)
    levels_div = np.linspace(-5,5,25)
    cf = ax.contourf(LAT2D, P2D, D.T,
                     levels=levels_div,
                     cmap='RdBu_r', extend='both')
    cs = ax.contour(LAT2D, P2D, D.T,
                    levels=np.arange(-5,6,1),
                    colors='k', linewidths=0.5)
    ax.clabel(cs, fmt='%d', inline=True, fontsize=8)
    cbar = fig.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label('EP flux divergence (m/s/day)')

    # 抽稀箭头：24×12 点
    nlat_q, nplev_q = 24, 12
    lat_idx  = np.linspace(0, len(lat)-1, nlat_q,  dtype=int)
    plev_idx = np.linspace(0, len(plev_hpa)-1, nplev_q, dtype=int)
    F1s = F1.isel(lat=lat_idx, plev=plev_idx)
    F2s = F2.isel(lat=lat_idx, plev=plev_idx)
    print(f" → Arrow points: {nlat_q}×{nplev_q}")

    # 构建 2D 网格
    X2d, Y2d = np.meshgrid(F1s.lat.values,
                           F1s.plev.values,
                           indexing='ij')
    w,h = ac.GetAxSize(fig, ax)
    print(f" Axes size (inches): {w:.2f}×{h:.2f}")

    # 物理缩放箭头（内部逻辑不变）
    ac.PlotEPfluxArrows(
        X2d, Y2d,
        F1s.values, F2s.values,
        fig, ax,
        xlim=[-90,90], ylim=[400,10],
        xscale='linear', yscale='log',
        invert_y=False,
        pivot='middle', scale=1e16
    )

    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(tag)
    fig.tight_layout()

    fout = os.path.join(outdir, f"epflux_{tag}.png")
    fig.savefig(fout, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(" → Saved:", fout)


# ----------------------------
# 主执行逻辑：WACCM 四组 + long‐run
# ----------------------------
year = "0008"
parent_dir = "/home/weiji/restart_exam/plots/epflux_speedtest/WACCM"
scenarios = [
    ("0008_Feb_couple",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     f"BWCN.e122.f19_g16.002_{year}/Feb/"
     f"BWCN.e122.f19_g16.002.{year}-02.*.nc"),
    ("0008_Mar_couple",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     f"BWCN.e122.f19_g16.002_{year}/Mar/"
     f"BWCN.e122.f19_g16.002.{year}-03.*.nc"),
    ("0008_Feb_nocouple",
     f"/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc"),
    ("0008_Mar_nocouple",
     f"/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc"),
]
wave_sets = [[1], [2], [3], [1,2]]

for tag, pattern in scenarios:
    print(f"\n=== Scenario: {tag} ===")
    outdir = os.path.join(parent_dir, tag)
    os.makedirs(outdir, exist_ok=True)

    files = sorted(glob.glob(pattern))
    # mask_and_scale=True 自动把 _FillValue 转 NaN
    ds_list = [xr.open_dataset(fp, mask_and_scale=True) for fp in files]
    ds = xr.concat(ds_list, dim='time')

    # 把异常值再一次屏蔽
    for var in ('U','V','T'):
        ds[var] = ds[var].where(np.abs(ds[var])<1e20)

    lat      = ds['lat'].values           # 96
    plev_hpa = ds['plev'].values/100.0    # 23→hPa
    u = ds['U'].values                    # (time,plev,lat,lon)
    v = ds['V'].values
    t = ds['T'].values
    print(" Data shapes:", u.shape, v.shape, t.shape)

    for waves in wave_sets:
        tag2 = f"{tag}_wave{'_'.join(map(str,waves))}"
        print(" • waves=", waves)
        nt = u.shape[0]
        ep1 = np.zeros((nt, len(plev_hpa), len(lat)))
        ep2 = np.zeros_like(ep1)
        d1  = np.zeros_like(ep1)
        d2  = np.zeros_like(ep1)
        for k in waves:
            e1,e2,dd1,dd2 = ac.ComputeEPfluxDiv(
                lat, plev_hpa, u, v, t,
                wave=k, do_ubar=False
            )
            ep1 += e1; ep2 += e2; d1 += dd1; d2 += dd2

        plot_vector_with_contour(lat, plev_hpa, ep1, ep2, d1, d2, outdir, tag2)

# long‐run 3.1–6.1
long_tag = f"{year}_long_3.1_6.1"
print(f"\n=== Scenario: {long_tag} ===")
outdir = os.path.join(parent_dir, long_tag)
os.makedirs(outdir, exist_ok=True)

ds_ref = xr.open_dataset(
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/"
    "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc",
    mask_and_scale=True
).sel(time=slice(f"{year}-03-01", f"{year}-06-01"))

for var in ('U','V','T'):
    ds_ref[var] = ds_ref[var].where(np.abs(ds_ref[var])<1e20)

lat      = ds_ref['lat'].values
plev_hpa = ds_ref['plev'].values/100.0
u = ds_ref['U'].values
v = ds_ref['V'].values
t = ds_ref['T'].values
print(" Data shapes:", u.shape, v.shape, t.shape)

for waves in wave_sets:
    tag2 = f"{long_tag}_wave{'_'.join(map(str,waves))}"
    print(" • waves=", waves)
    nt = u.shape[0]
    ep1 = np.zeros((nt, len(plev_hpa), len(lat)))
    ep2 = np.zeros_like(ep1)
    d1  = np.zeros_like(ep1)
    d2  = np.zeros_like(ep1)
    for k in waves:
        e1,e2,dd1,dd2 = ac.ComputeEPfluxDiv(
            lat, plev_hpa, u, v, t,
            wave=k, do_ubar=False
        )
        ep1 += e1; ep2 += e2; d1 += dd1; d2 += dd2

    plot_vector_with_contour(lat, plev_hpa, ep1, ep2, d1, d2, outdir, tag2)

print("\nAll WACCM experiments done!")
